In [1]:
#general imports that we will need will almost always use - it is a good practice to import all libraries at the beginning of the notebook or script
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

# data partition
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

#filter methods
# spearman 
# chi-square
import scipy.stats as stats
from scipy.stats import chi2_contingency

#wrapper methods
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import lasso_path, SGDRegressor


# embedded methods
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.preprocessing import MinMaxScaler


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.calibration import LabelEncoder
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor


from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import ElasticNet
from sklearn.compose import TransformedTargetRegressor


#set random seed for reproducibility
RSEED = 42
np.random.seed(RSEED)


In [2]:
# path to folder
df_train = pd.read_csv("project_data/train.csv", delimiter=',', header=0, decimal='.', quotechar='"')

In [3]:
#dividing into X and y, as well as validation set and training set
X_test = pd.read_csv("project_data/test.csv", delimiter=',', header=0, decimal='.', quotechar='"')


In [4]:
df_train.drop('paintQuality%', axis=1, inplace=True)
X_test.drop('paintQuality%', axis=1, inplace=True)

In [5]:
#correcting the data types
df_train['year']=pd.to_datetime(df_train['year'], format='%Y')
df_train["year"] = df_train["year"].dt.year
X_test['year']=pd.to_datetime(X_test['year'], format='%Y')
X_test["year"] = X_test["year"].dt.year

df_train['previousOwners'] = df_train['previousOwners'].apply(lambda x: int(x) if pd.notna(x) else x)
X_test['previousOwners'] = X_test['previousOwners'].apply(lambda x: int(x) if pd.notna(x) else x)

df_train['hasDamage'] = df_train['hasDamage'].apply(lambda x: False if pd.isna(x) else True)
X_test['hasDamage'] = X_test['hasDamage'].apply(lambda x: False if pd.isna(x) else True)


In [6]:
df_train.set_index('carID', inplace = True)
X_test.set_index('carID', inplace = True)

In [7]:
df_train['Brand'].unique()

array(['VW', 'Toyota', 'Audi', 'Ford', 'BMW', 'Skoda', 'Opel', 'Mercedes',
       'FOR', 'mercedes', 'Hyundai', 'w', 'ord', 'MW', 'bmw', nan,
       'yundai', 'BM', 'Toyot', 'udi', 'Ope', 'AUDI', 'V', 'opel', 'pel',
       'For', 'pe', 'Mercede', 'audi', 'MERCEDES', 'OPEL', 'koda', 'FORD',
       'Hyunda', 'W', 'Aud', 'vw', 'hyundai', 'skoda', 'ford', 'TOYOTA',
       'ercedes', 'oyota', 'toyota', 'SKODA', 'Skod', 'HYUNDAI', 'kod',
       'v', 'for', 'SKOD', 'aud', 'KODA', 'PEL', 'yunda', 'or', 'UDI',
       'OYOTA', 'HYUNDA', 'mw', 'OPE', 'mercede', 'ERCEDES', 'ercede',
       'TOYOT', 'MERCEDE', 'ORD', 'ud', 'ope', 'AUD', 'hyunda', 'skod',
       'toyot'], dtype=object)

In [8]:
from fuzzywuzzy import process, fuzz

In [9]:
import pandas as pd
import numpy as np
from difflib import SequenceMatcher, get_close_matches
from collections import Counter

def similarity_ratio(a, b):
    """Calcula similaridade entre duas strings (0-1)"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def clean_categorical_column(df, column_name, 
                             case_threshold=0.95,
                             truncation_threshold=0.85,
                             min_length=2,
                             aggressive_short=True,
                             invalid_values=None,
                             show_changes=True):
    """
    Limpeza inteligente de colunas categóricas usando difflib
    
    Parameters:
    -----------
    df : DataFrame
    column_name : str - nome da coluna
    case_threshold : float - threshold para variações de case (0.95 = muito restritivo)
    truncation_threshold : float - threshold para truncamentos (0.85 = moderado)
    min_length : int - comprimento mínimo válido
    aggressive_short : bool - ativar correção agressiva de valores curtos
    invalid_values : list - valores a converter para NaN (ex: ['unknown', 'Other'])
    show_changes : bool - mostrar mudanças feitas
    
    Returns:
    --------
    df : DataFrame limpo
    """
    
    original_col = df[column_name].copy()
    
    # CRÍTICO: Remover espaços extras e normalizar
    df[column_name] = df[column_name].astype(str).str.strip()
    df[column_name] = df[column_name].replace('', np.nan)  # Strings vazias → NaN
    df[column_name] = df[column_name].replace('nan', np.nan)  # String 'nan' → NaN real
    
    # Converter valores inválidos para NaN antes de começar
    if invalid_values:
        invalid_count = 0
        for inv_val in invalid_values:
            mask = df[column_name].str.lower() == inv_val.lower()
            invalid_count += mask.sum()
            df[column_name] = df[column_name].mask(mask, np.nan)
        
        if invalid_count > 0:
            print(f"🗑️  Convertidos {invalid_count} valores inválidos para NaN: {invalid_values}")
    
    values = df[column_name].dropna()
    unique_vals = values.unique()
    value_counts = values.value_counts()
    
    print(f"\n{'='*60}")
    print(f"🧹 Limpando coluna: {column_name}")
    print(f"{'='*60}")
    print(f"Valores únicos originais: {len(unique_vals)}")
    
    # ===========================================
    # FASE 1: Normalizar variações de case e espaços
    # ===========================================
    case_mapping = {}
    lower_groups = {}
    
    for val in unique_vals:
        # Normalizar: lowercase + remover espaços múltiplos
        normalized = ' '.join(val.lower().split())
        
        if normalized not in lower_groups:
            lower_groups[normalized] = []
        lower_groups[normalized].append(val)
    
    for normalized, variants in lower_groups.items():
        if len(variants) > 1:
            # Escolher versão canônica: sem espaços extras > Title Case > mais comum > mais longa
            canonical = max(variants, key=lambda x: (
                len(x.split()) == len(x.strip().split()),  # Sem espaços extras
                x[0].isupper() and not x.isupper(),         # Title Case
                value_counts.get(x, 0),                     # Frequência
                len(x)                                       # Comprimento
            ))
            
            for variant in variants:
                if variant != canonical:
                    case_mapping[variant] = canonical
    
    df[column_name] = df[column_name].replace(case_mapping)
    print(f"✓ Fase 1: {len(case_mapping)} correções de case/espaços")
    
    # ===========================================
    # FASE 2: Corrigir truncamentos óbvios
    # ===========================================
    values = df[column_name].dropna().astype(str)
    unique_vals = values.unique()
    value_counts = values.value_counts()
    
    truncation_mapping = {}
    processed = set()
    
    # Ordenar por comprimento (curtos primeiro)
    sorted_vals = sorted(unique_vals, key=lambda x: (len(x), x))
    
    for short_val in sorted_vals:
        if short_val in processed or len(short_val) < min_length:
            continue
        
        # Procurar candidatos mais longos
        candidates = [v for v in unique_vals 
                     if len(v) > len(short_val) and v not in processed]
        
        if not candidates:
            continue
        
        # Usar get_close_matches do difflib
        matches = get_close_matches(
            short_val, 
            candidates, 
            n=3,
            cutoff=truncation_threshold
        )
        
        if matches:
            best_match = matches[0]
            
            # Condições para truncamento:
            sim = similarity_ratio(short_val, best_match)
            min_prefix = min(len(short_val), max(3, int(len(best_match) * 0.6)))
            is_prefix = best_match.lower().startswith(short_val[:min_prefix].lower())
            
            # Aceitar se: alta similaridade OU é claramente um prefixo
            if (sim >= truncation_threshold and is_prefix) or sim >= 0.95:
                short_count = value_counts.get(short_val, 0)
                match_count = value_counts.get(best_match, 0)
                
                if match_count >= short_count or len(short_val) < len(best_match) * 0.7:
                    truncation_mapping[short_val] = best_match
                    processed.add(short_val)
    
    df[column_name] = df[column_name].replace(truncation_mapping)
    print(f"✓ Fase 2: {len(truncation_mapping)} correções de truncamento")
    
    # ===========================================
    # FASE 3: Corrigir valores muito curtos (agressivo)
    # ===========================================
    if aggressive_short:
        values = df[column_name].dropna().astype(str)
        unique_vals = values.unique()
        value_counts = values.value_counts()
        
        short_mapping = {}
        
        # Regras específicas para casos conhecidos (Brand)
        specific_rules = {
            'W': 'VW', 'V': 'VW',
            'MW': 'BMW', 'BM': 'BMW',
            'ercedes': 'Mercedes',
            'oyota': 'Toyota',
            'koda': 'Skoda',
            'yundai': 'Hyundai'
        }
        
        # Regras específicas para transmission e fuelType
        if column_name.lower() in ['transmission', 'fueltype', 'fuel_type', 'fuel']:
            specific_rules.update({
                'anual': 'Manual',
                'nknown': 'unknown',
                'nknow': 'unknown',
                'utomatic': 'Automatic',
                'emi-Auto': 'Semi-Auto',
                'etrol': 'Petrol',
                'iesel': 'Diesel',
                'ybrid': 'Hybrid',
                'ther': 'Other'
            })
        
        for short, full in specific_rules.items():
            if short in unique_vals and full in unique_vals:
                short_mapping[short] = full
        
        # Para outros valores muito curtos (1-3 chars), procurar versão completa
        very_short = [v for v in unique_vals if len(v) <= 3 and v not in short_mapping]
        normal_vals = [v for v in unique_vals if len(v) > 3]
        
        for short in very_short:
            # Procurar valores que contenham este short como substring
            candidates = [v for v in normal_vals 
                         if short.lower() in v.lower() 
                         or v.lower().startswith(short.lower())]
            
            if candidates:
                # Pegar o mais comum
                best = max(candidates, key=lambda x: value_counts.get(x, 0))
                
                # Verificar se faz sentido
                if similarity_ratio(short, best) >= 0.5:
                    short_mapping[short] = best
        
        df[column_name] = df[column_name].replace(short_mapping)
        print(f"✓ Fase 3: {len(short_mapping)} correções agressivas de valores curtos")
    
    # ===========================================
    # FASE 4: Detectar valores suspeitos
    # ===========================================
    values = df[column_name].dropna().astype(str)
    unique_vals = values.unique()
    
    # Valores muito curtos que não foram corrigidos
    short_vals = [v for v in unique_vals if len(v) < min_length]
    
    # Valores que aparecem muito pouco
    rare_vals = [v for v, count in value_counts.items() 
                 if count == 1 and len(v) < 5]
    
    if short_vals:
        print(f"\n⚠️  Valores suspeitos (muito curtos): {short_vals[:15]}")
    
    if rare_vals and len(rare_vals) <= 20:
        print(f"⚠️  Valores raros (aparecem 1x): {rare_vals[:15]}")
    
    # ===========================================
    # RESUMO
    # ===========================================
    final_unique = df[column_name].nunique()
    total_corrections = len(case_mapping) + len(truncation_mapping)
    if aggressive_short:
        total_corrections += len(short_mapping)
    
    print(f"\n{'─'*60}")
    print(f"📊 Resumo:")
    print(f"   Valores únicos: {len(unique_vals)} → {final_unique}")
    print(f"   Total de correções: {total_corrections}")
    print(f"   Redução: {len(unique_vals) - final_unique} valores")
    print(f"{'='*60}\n")
    
    # Mostrar algumas mudanças
    if show_changes and total_corrections > 0:
        print("📝 Exemplos de correções feitas:")
        all_mappings = {**case_mapping, **truncation_mapping}
        if aggressive_short:
            all_mappings.update(short_mapping)
        for old, new in list(all_mappings.items())[:10]:
            count = (original_col == old).sum()
            print(f"   '{old}' → '{new}' ({count} ocorrências)")
        if len(all_mappings) > 10:
            print(f"   ... e mais {len(all_mappings) - 10} correções")
        print()
    
    return df


def preview_unique_values(df, column_name, top_n=30):
    """Preview dos valores únicos mais comuns"""
    print(f"\n📋 Top {top_n} valores em '{column_name}':")
    print(df[column_name].value_counts().head(top_n))


def find_potential_duplicates(df, column_name, threshold=0.85):
    """
    Encontra pares de valores que podem ser duplicatas
    """
    values = df[column_name].dropna().unique()
    
    print(f"\n🔍 Procurando possíveis duplicatas em '{column_name}'...")
    
    potential_dupes = []
    
    for i, val1 in enumerate(values):
        # Usar get_close_matches para encontrar similares
        similar = get_close_matches(
            val1, 
            values[i+1:],  # Apenas valores ainda não comparados
            n=5,
            cutoff=threshold
        )
        
        for val2 in similar:
            sim = similarity_ratio(val1, val2)
            count1 = (df[column_name] == val1).sum()
            count2 = (df[column_name] == val2).sum()
            potential_dupes.append((val1, val2, sim, count1, count2))
    
    if potential_dupes:
        print(f"\n⚠️  Encontrados {len(potential_dupes)} pares suspeitos:")
        print(f"{'Valor 1':<25} {'Valor 2':<25} {'Similaridade':<12} {'Count1':<8} {'Count2'}")
        print("─" * 85)
        for v1, v2, sim, c1, c2 in sorted(potential_dupes, key=lambda x: x[2], reverse=True)[:20]:
            print(f"{v1:<25} {v2:<25} {sim:.2%}{'':<9} {c1:<8} {c2}")
    else:
        print("✓ Nenhuma duplicata óbvia encontrada!")
    
    return potential_dupes


# ============================================
# EXEMPLO DE USO
# ============================================

# Limpeza básica
df_train = clean_categorical_column(df_train, 'Brand')
# df_train = clean_categorical_column(df_train, 'Model')

# Com parâmetros customizados para ser mais agressivo
df_train = clean_categorical_column(
    df_train, 
    'model',
    case_threshold=0.95,
    truncation_threshold=0.80,  # Mais permissivo
    min_length=2
)
df_train = clean_categorical_column(
    df_train, 
    'transmission',
    aggressive_short=True,
    truncation_threshold=0.80,
    invalid_values=['unknown', 'Other','unknow', 'nknown', 'nknow']
)

# Limpar fuelType - remover unknown e Other
df_train = clean_categorical_column(
    df_train, 
    'fuelType',
    aggressive_short=True,
    truncation_threshold=0.80,
    invalid_values=['unknown', 'Other','ther', 'Othe'])


# Limpeza básica
X_test = clean_categorical_column(X_test, 'Brand')
# df_train = clean_categorical_column(df_train, 'Model')

# Com parâmetros customizados para ser mais agressivo
X_test = clean_categorical_column(
    X_test, 
    'model',
    case_threshold=0.95,
    truncation_threshold=0.80,  # Mais permissivo
    min_length=2
)
X_test = clean_categorical_column(
    X_test, 
    'transmission',
    aggressive_short=True,
    truncation_threshold=0.80,
    invalid_values=['unknown', 'Other','unknow', 'nknown', 'nknow']
)

# Limpar fuelType - remover unknown e Other
X_test = clean_categorical_column(
    X_test, 
    'fuelType',
    aggressive_short=True,
    truncation_threshold=0.80,
    invalid_values=['unknown', 'Other','ther', 'Othe'])

# Verificar resultado
# preview_unique_values(df_train, 'Brand')
# preview_unique_values(df_train, 'Model', top_n=50)

# Procurar mais duplicatas potenciais
# find_potential_duplicates(df_train, 'Model', threshold=0.82)


🧹 Limpando coluna: Brand
Valores únicos originais: 72
✓ Fase 1: 39 correções de case/espaços
✓ Fase 2: 10 correções de truncamento
✓ Fase 3: 14 correções agressivas de valores curtos

────────────────────────────────────────────────────────────
📊 Resumo:
   Valores únicos: 9 → 9
   Total de correções: 63
   Redução: 0 valores

📝 Exemplos de correções feitas:
   'vw' → 'VW' (193 ocorrências)
   'TOYOTA' → 'Toyota' (82 ocorrências)
   'toyota' → 'Toyota' (84 ocorrências)
   'AUDI' → 'Audi' (144 ocorrências)
   'audi' → 'Audi' (135 ocorrências)
   'FORD' → 'Ford' (316 ocorrências)
   'ford' → 'Ford' (307 ocorrências)
   'bmw' → 'BMW' (134 ocorrências)
   'skoda' → 'Skoda' (93 ocorrências)
   'SKODA' → 'Skoda' (72 ocorrências)
   ... e mais 53 correções


🧹 Limpando coluna: model
Valores únicos originais: 546
✓ Fase 1: 250 correções de case/espaços
✓ Fase 2: 97 correções de truncamento
✓ Fase 3: 5 correções agressivas de valores curtos

⚠️  Valores suspeitos (muito curtos): ['Q', 'A', 'X'

In [10]:
df_train['transmission'].unique()

array(['Semi-Auto', 'Manual', 'Automatic', nan], dtype=object)

In [11]:
df_train['fuelType'].unique()

array(['Petrol', 'Diesel', 'Hybrid', nan, 'Electric'], dtype=object)

In [12]:
df_train.loc[df_train["tax"] < 0, "tax"] = np.nan
df_train.loc[df_train["engineSize"] <= 0, "engineSize"] = np.nan
df_train.loc[df_train["mileage"] < 0, "mileage"] = np.nan
df_train.loc[df_train["mpg"] <= 0, "mpg"] = np.nan
df_train.loc[df_train["previousOwners"] <= 0, "previousOwners"] = np.nan

X_test.loc[X_test["tax"] < 0, "tax"] = np.nan
X_test.loc[X_test["engineSize"] <= 0, "engineSize"] = np.nan
X_test.loc[X_test["mileage"] < 0, "mileage"] = np.nan
X_test.loc[X_test["mpg"] <= 0, "mpg"] = np.nan
X_test.loc[X_test["previousOwners"] <= 0, "previousOwners"] = np.nan

In [13]:
df_train["has_reportedDamage"]=df_train["hasDamage"].map(lambda x: 1 if x==True else 0)
X_test["has_reportedDamage"]=X_test["hasDamage"].map(lambda x: 1 if x==True else 0)

# dropping the original 'hasDamage' column bc of redundancy
df_train.drop('hasDamage', axis=1, inplace=True)
X_test.drop('hasDamage', axis=1, inplace=True)

In [14]:
cat_features=["Brand", "model", "fuelType", "transmission"]
metric_features=df_train.columns.drop(cat_features).tolist()

df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75973 entries, 69512 to 15795
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Brand               74452 non-null  object 
 1   model               74456 non-null  object 
 2   year                74482 non-null  float64
 3   price               75973 non-null  int64  
 4   transmission        73710 non-null  object 
 5   mileage             74141 non-null  float64
 6   fuelType            74295 non-null  object 
 7   tax                 67691 non-null  float64
 8   mpg                 68011 non-null  float64
 9   engineSize          74193 non-null  float64
 10  previousOwners      59173 non-null  float64
 11  has_reportedDamage  75973 non-null  int64  
dtypes: float64(6), int64(2), object(4)
memory usage: 7.5+ MB


In [15]:
X = df_train.drop('price', axis = 1)
y = df_train['price']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, shuffle=True
)


metric_features.remove('price')
print(metric_features)

['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'has_reportedDamage']


In [16]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neighbors import NearestNeighbors


In [17]:


def impute_missing_values_hybrid(X_train, X_val, X_test):
    '''
    Hybrid intelligent imputation:
    1. Simple categorical: model, Brand (rules + mode)
    2. Conditional categorical: fuelType, transmission (mode by group)
    3. Binary flags: has_damage, has_reported_damage (mode)
    4. Correlated numerical: IterativeImputer (MICE)
    5. Optional flags to indicate imputed values
    6. Plausibility clipping
    Args:
        X_train: Training feature set.
        X_val: Validation feature set.
        X_test: Test feature set.
    '''
    
    X_tr = X_train.copy()
    X_v = X_val.copy()
    X_te = X_test.copy()
    
    print("="*80)
    print("HYBRID IMPUTATION PIPELINE")
    print("="*80)
    
    
# =========================================================================
    # STEP 1: MODEL (brand mode if Brand known, else global mode)
    # =========================================================================
    '''
    print("\n[1/6] MODEL - brand-aware mode + global fallback")

    
    global_mode_model = X_tr["model"].mode()[0] if len(X_tr["model"].mode()) > 0 else "unknown"

   
    n_missing_train = X_tr["model"].isna().sum()

    
    brand_to_model_mode = (
        X_tr.dropna(subset=["Brand", "model"])
            .groupby("Brand")["model"]
            .agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else None)
            .to_dict()
    )

    def fill_model(df):
        """Fill missing model values.
        Args:
            df: DataFrame to process.
        """
        miss_model = df["model"].isna()

        
        has_brand = df["Brand"].notna()
        idx_brand = df.index[miss_model & has_brand]
        df.loc[idx_brand, "model"] = df.loc[idx_brand, "Brand"].map(brand_to_model_mode)

      
        df["model"] = df["model"].fillna(global_mode_model)

    
    fill_model(X_tr)
    fill_model(X_v)
    fill_model(X_te)

    print(f"  Global mode: '{global_mode_model}'")
    print(f"  Imputed - Train: {n_missing_train}, Val: {X_val['model'].isna().sum()}, "
        f"Test: {X_test['model'].isna().sum()}")
    # =========================================================================
    # STEP 2: BRAND (inferred from model, then mode)
    # =========================================================================
    print("\n[2/6] BRAND - inferred from model + learned mapping")
    
    # Create model->Brand dictionary from known data
    model_to_brand_map = (
        X_tr.dropna(subset=['Brand', 'model'])
        .groupby('model')['Brand']
        .agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else None)
        .to_dict()
    )
    
    # Fallback: hardcoded lists for cases not in data
    toyota = ["yaris", "aygo", "corolla", "chr", "avensis", "prius", "rav4", "hilux", 
              "verso", "supra", "landcruiser", "camry", "proaceverso", "urbancruiser", 
              "auris", "gt86"]
    ford = ["focus", "fiesta", "mondeo", "kuga", "galaxy", "smax", "bmax", "ecosport", 
            "puma", "tourneocustom", "tourneoconnect", "grandtourneoconnect", "cmax", 
            "grandcmax", "edge", "mustang", "fusion", "streetka", "ranger", "escort", 
            "ka", "ka+"]
    opel = ["corsa", "mokkax", "astra", "insignia", "mokka", "zafira", "viva", "meriva", 
            "adam", "combolife", "crosslandx", "grandlandx", "gtc", "antara", "vivaro", 
            "vectra", "agila", "tigra", "cascada", "ampera"]
    vw = ["golf", "golfsv", "polo", "passat", "tiguan", "tiguanallspace", "touran", 
          "touareg", "troc", "tcross", "arteon", "sharan", "jetta", "cc", "caravelle", 
          "california", "caddy", "caddymaxi", "beetle", "scirocco", "up", "amarok", "eos", "fox"]
    audi = ["a1", "a2", "a3", "a4", "a5", "a6", "a7", "a8", "q2", "q3", "q5", "q7", 
            "q8", "s3", "s4", "s5", "s8", "rs3", "rs4", "rs5", "rs6", "sq5", "sq7", "tt", "r8"]
    mercedes = ["aclass", "bclass", "cclass", "eclass", "sclass", "claclass", "clsclass", 
                "glaclass", "glbclass", "glcclass", "gleclass", "glsclass", "glclass", 
                "gclass", "vclass", "xclass", "slclass", "slkclass", "mclass", "slc", 
                "clk", "clclass", "clcclass", "mercedes200", "mercedes220", "mercedes230"]
    skoda = ["fabia", "octavia", "superb", "karoq", "kodiaq", "kamiq", "yeti", 
             "yetioutdoor", "scala", "rapid", "citigo", "roomster"]
    hyundai = ["i10", "i20", "i30", "i40", "i800", "ioniq", "kona", "tucson", "santafe", 
               "getz", "ix20", "ix35", "veloster", "accent", "terracan"]
    bmw_models = ["series1", "series2", "series3", "series4", "series5", "series6", 
                  "series7", "series8", "x1", "x2", "x3", "x4", "x5", "x6", "x7", 
                  "z3", "z4", "m2", "m3", "m4", "m5", "m6", "iq"]
    seat_models = ["leon", "ateca", "toledo", "arona", "ibiza", "alhambra"]
    
    def infer_brand_smart(model_val):
        """Infer Brand from model using learned mapping and hardcoded lists.
        Args:
            model_val: The model value to infer the brand for.
        """
        if pd.isna(model_val):
            return None
        
        # First try learned mapping
        if model_val in model_to_brand_map:
            return model_to_brand_map[model_val]
        
        # Fallback to hardcoded lists
        m = str(model_val).lower()
        if m in toyota: return "toyota"
        if m in ford: return "ford"
        if m in opel: return "opel"
        if m in vw: return "vw"
        if m in audi: return "audi"
        if m in bmw_models: return "bmw"
        if m in mercedes: return "mercedes"
        if m in skoda: return "skoda"
        if m in hyundai: return "hyundai"
        if m in seat_models: return "seat"
        if m == "kadjar": return "renault"
        if m == "shuttle": return "honda"
        return None
    
    # Apply inference
    n_missing_brand = X_tr["Brand"].isna().sum()
    for df in [X_tr, X_v, X_te]:
        mask_nan = df["Brand"].isna()
        df.loc[mask_nan, "Brand"] = df.loc[mask_nan, "model"].apply(infer_brand_smart)
    
    # Global mode for remaining
    global_mode_brand = X_tr["Brand"].mode()[0] if len(X_tr["Brand"].mode()) > 0 else "ford"
    X_tr["Brand"].fillna(global_mode_brand, inplace=True)
    X_v["Brand"].fillna(global_mode_brand, inplace=True)
    X_te["Brand"].fillna(global_mode_brand, inplace=True)
    
    print(f"  Learned mapping: {len(model_to_brand_map)} models")
    print(f"  Imputed - Train: {n_missing_brand}, Val: {X_val['Brand'].isna().sum()}, "
          f"Test: {X_test['Brand'].isna().sum()}")
    
    # =========================================================================
    # STEP 3: CONDITIONAL CATEGORICAL (fuelType, transmission)
    # =========================================================================
    print("\n[3/6] FUELTYPE & TRANSMISSION - mode by group")
    
    # fuelType by Brand
    mode_fueltype_brand = (
        X_tr.groupby("Brand")["fuelType"]
        .apply(lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan)
    )
    global_mode_fueltype = X_tr["fuelType"].mode()[0] if len(X_tr["fuelType"].mode()) > 0 else "Petrol"
    
    def fill_fueltype(row):
        """Fill missing fuelType based on Brand mode, else global mode.
        Args:
            row: DataFrame row to process.
        """
        if pd.notna(row["fuelType"]):
            return row["fuelType"]
        val = mode_fueltype_brand.get(row["Brand"], global_mode_fueltype)
        return val if pd.notna(val) else global_mode_fueltype
    
    n_missing_fuel = X_tr["fuelType"].isna().sum()
    X_tr["fuelType"] = X_tr.apply(fill_fueltype, axis=1)
    X_v["fuelType"] = X_v.apply(fill_fueltype, axis=1)
    X_te["fuelType"] = X_te.apply(fill_fueltype, axis=1)
    
    # transmission by Brand + fuelType
    mode_transmission_brandfuel = (
        X_tr.groupby(["Brand", "fuelType"])["transmission"]
        .apply(lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan)
    )
    mode_transmission_brand = (
        X_tr.groupby("Brand")["transmission"]
        .apply(lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan)
    )
    global_mode_transmission = X_tr["transmission"].mode()[0] if len(X_tr["transmission"].mode()) > 0 else "Manual"
    
    def fill_transmission(row):
        """Fill missing transmission based on (Brand, fuelType) mode,
        then Brand mode, else global mode.
        Args:
            row: DataFrame row to process.
        """
        if pd.notna(row["transmission"]):
            return row["transmission"]
        val = mode_transmission_brandfuel.get((row["Brand"], row["fuelType"]))
        if pd.isna(val):
            val = mode_transmission_brand.get(row["Brand"], global_mode_transmission)
        return val if pd.notna(val) else global_mode_transmission
    
    n_missing_trans = X_tr["transmission"].isna().sum()
    X_tr["transmission"] = X_tr.apply(fill_transmission, axis=1)
    X_v["transmission"] = X_v.apply(fill_transmission, axis=1)
    X_te["transmission"] = X_te.apply(fill_transmission, axis=1)
    
    print(f"  fuelType imputed - Train: {n_missing_fuel}")
    print(f"  transmission imputed - Train: {n_missing_trans}")
    
    # =========================================================================
    # STEP 3.5: BINARY FLAGS (has_damage, has_reported_damage)
    # =========================================================================
    print("\n[3.5/6] BINARY FLAGS - has_reported_damage")
    
    for col in ['has_reported_damage']:
        if col in X_tr.columns:
            mode_val = X_tr[col].mode()[0] if len(X_tr[col].mode()) > 0 else 0
            n_missing_train = X_tr[col].isna().sum()
            n_missing_val = X_v[col].isna().sum()
            n_missing_test = X_te[col].isna().sum()
            
            X_tr[col].fillna(mode_val, inplace=True)
            X_v[col].fillna(mode_val, inplace=True)
            X_te[col].fillna(mode_val, inplace=True)
            
            if n_missing_train > 0 or n_missing_val > 0 or n_missing_test > 0:
                print(f"  {col} - mode: {mode_val}, imputed Train: {n_missing_train}, "
                      f"Val: {n_missing_val}, Test: {n_missing_test}")
    
    # =========================================================================
    # STEP 4: ENSURE KNOWN CATEGORICAL VALUES (before MICE)
    # =========================================================================
    print("\n[4/6] SYNCHRONIZATION - force known categorical values")
    
    cat_cols_to_sync = ['Brand', 'model', 'fuelType', 'transmission']
    
    for col in cat_cols_to_sync:
        if col in X_tr.columns:
            # Get known values (excluding NaN)
            known_values = set(X_tr[col].dropna().unique())
            mode_val = X_tr[col].mode()[0]
            
            # Val: replace unknown with mode (only non-null values)
            mask_unknown_val = X_v[col].notna() & (~X_v[col].isin(known_values))
            n_unknown_val = mask_unknown_val.sum()
            if n_unknown_val > 0:
                X_v.loc[mask_unknown_val, col] = mode_val
                print(f"  {col} - Val: {n_unknown_val} unknown values -> '{mode_val}'")
            
            # Test: same
            mask_unknown_test = X_te[col].notna() & (~X_te[col].isin(known_values))
            n_unknown_test = mask_unknown_test.sum()
            if n_unknown_test > 0:
                X_te.loc[mask_unknown_test, col] = mode_val
                print(f"  {col} - Test: {n_unknown_test} unknown values -> '{mode_val}'")
    '''
    # =========================================================================
    # STEP 5: CORRELATED NUMERICAL - IterativeImputer (MICE)
    # =========================================================================
    print("\n[5/6] NUMERICAL - IterativeImputer (MICE)")
    
    numeric_cols = ['year', 'engineSize', 'mileage', 'mpg', 'tax', 'previousOwners']
    
    # Check which have missing
    numeric_cols_with_missing = [col for col in numeric_cols 
                                  if X_tr[col].isna().sum() > 0]
    
    if numeric_cols_with_missing:
        print(f"  Columns to impute: {numeric_cols_with_missing}")
        
        # Prepare data for imputer
        # Convert categorical to numeric codes temporarily
        cat_cols = ['Brand', 'model', 'fuelType', 'transmission']
        
        # Create temporary copies
        X_tr_temp = X_tr.copy()
        X_v_temp = X_v.copy()
        X_te_temp = X_te.copy()
        
        # Temporary label encoding
        label_mappings = {}
        for col in cat_cols:
            if col in X_tr_temp.columns:
                # Create mapping from train (excluding NaN)
                unique_vals = X_tr_temp[col].dropna().unique()
                mapping = {val: idx for idx, val in enumerate(unique_vals)}
                label_mappings[col] = mapping
                
                # Apply mapping (unknown values remain as NaN)
                X_tr_temp[col] = X_tr_temp[col].map(mapping)
                X_v_temp[col] = X_v_temp[col].map(mapping)
                X_te_temp[col] = X_te_temp[col].map(mapping)
        
        # Select features for imputer
        features_for_imputation = cat_cols + numeric_cols
        features_for_imputation = [f for f in features_for_imputation if f in X_tr_temp.columns]
        
        # Configure and train imputer
        imputer = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=10, max_depth=10, random_state=42),
            max_iter=10,
            random_state=42,
            verbose=0
        )
        
        # Fit on train
        X_tr_imputed = imputer.fit_transform(X_tr_temp[features_for_imputation])
        X_v_imputed = imputer.transform(X_v_temp[features_for_imputation])
        X_te_imputed = imputer.transform(X_te_temp[features_for_imputation])
        
        # Replace only imputed numerical columns
        for i, col in enumerate(numeric_cols):
            if col in features_for_imputation:
                idx = features_for_imputation.index(col)
                X_tr[col] = X_tr_imputed[:, idx]
                X_v[col] = X_v_imputed[:, idx]
                X_te[col] = X_te_imputed[:, idx]
        
        print(f"  IterativeImputer applied successfully")
    else:
        print(f"  No numerical columns with missing values")
    
    # =========================================================================
    # STEP 6: VALIDATION AND CORRECTIONS
    # =========================================================================
    print("\n[6/6] VALIDATION - checking logical limits")
    
    # Sanity corrections
    if 'year' in X_tr.columns:
        for df in [X_tr, X_v, X_te]:
            df['year'] = df['year'].clip(upper=2025)
    
    if 'engineSize' in X_tr.columns:
        for df in [X_tr, X_v, X_te]:
            df['engineSize'] = df['engineSize'].clip(lower=0.5)
    
    if 'mileage' in X_tr.columns:
        for df in [X_tr, X_v, X_te]:
            df['mileage'] = df['mileage'].clip(lower=0, upper=500000)
    
    if 'mpg' in X_tr.columns:
        for df in [X_tr, X_v, X_te]:
            df['mpg'] = df['mpg'].clip(lower=10, upper=200)
    
    if 'tax' in X_tr.columns:
        for df in [X_tr, X_v, X_te]:
            df['tax'] = df['tax'].clip(lower=0, upper=1000)
    
    if 'previousOwners' in X_tr.columns:
        for df in [X_tr, X_v, X_te]:
            df['previousOwners'] = df['previousOwners'].clip(lower=0, upper=10).round()
    
    print(f"  Limits applied")
    
    # =========================================================================
    # FINAL REPORT
    # =========================================================================
    print("\n" + "="*80)
    print("IMPUTATION COMPLETED")
    print("="*80)
    
    print("\nFinal missing values:")
    print(f"  Train: {X_tr.isna().sum().sum()}")
    print(f"  Val:   {X_v.isna().sum().sum()}")
    print(f"  Test:  {X_te.isna().sum().sum()}")
    
    if X_tr.isna().sum().sum() > 0:
        print("\nColumns with remaining NaNs in Train:")
        print(X_tr.isna().sum()[X_tr.isna().sum() > 0])
    
    if X_v.isna().sum().sum() > 0:
        print("\nColumns with remaining NaNs in Val:")
        print(X_v.isna().sum()[X_v.isna().sum() > 0])
    
    if X_te.isna().sum().sum() > 0:
        print("\nColumns with remaining NaNs in Test:")
        print(X_te.isna().sum()[X_te.isna().sum() > 0])
    
    return X_tr, X_v, X_te


def impute_knn_categorical(X_train, X_val, X_test):
    # =========================================================================
    # STEP 3.25: kNN CATEGORICAL (leakage-safe)
    # =========================================================================
    print("\n[3.25/6] kNN CATEGORICAL IMPUTATION (train-only)")

    

    # Categóricas a refinar com kNN (opcional)
    knn_cat_cols = ['fuelType', 'transmission', 'Brand', 'model']
    knn_num_cols = ['year', 'engineSize', 'mileage', 'mpg', 'tax']

    knn_cat_cols = [c for c in knn_cat_cols if c in X_train.columns]
    knn_num_cols = [c for c in knn_num_cols if c in X_train.columns]

    if knn_cat_cols and knn_num_cols:

        # Escalar numéricas (fit só no treino)
        scaler_knn = StandardScaler()
        X_tr_num_scaled = scaler_knn.fit_transform(X_train[knn_num_cols])
        X_v_num_scaled  = scaler_knn.transform(X_val[knn_num_cols])
        X_te_num_scaled = scaler_knn.transform(X_test[knn_num_cols])

        # Base kNN: apenas linhas completas do treino
        mask_complete = X_train[knn_cat_cols + knn_num_cols].notna().all(axis=1)

        X_knn_base = X_train.loc[mask_complete, knn_cat_cols + knn_num_cols].copy()
        X_knn_num_base = scaler_knn.transform(X_knn_base[knn_num_cols])

        # Codificar categorias como códigos (apenas treino)
        cat_maps = {}
        for col in knn_cat_cols:
            X_knn_base[col] = X_knn_base[col].astype('category')
            cat_maps[col] = X_knn_base[col].cat.categories
            X_knn_base[col] = X_knn_base[col].cat.codes

        X_knn_matrix = np.hstack([X_knn_num_base, X_knn_base[knn_cat_cols].values])

        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(X_knn_matrix)

        def knn_impute(df, df_num_scaled, name):
            n_imputed = 0

            for col in knn_cat_cols:
                for idx in df[df[col].isna()].index:

                    row_num = df_num_scaled[df.index.get_loc(idx)].reshape(1, -1)

                    dummy_cat = np.zeros((1, len(knn_cat_cols)))
                    row_vec = np.hstack([row_num, dummy_cat])

                    _, neighbors = knn.kneighbors(row_vec)
                    neigh_vals = X_knn_base.iloc[neighbors[0]][col]
                    neigh_vals = neigh_vals[neigh_vals >= 0]

                    if len(neigh_vals) > 0:
                        code = neigh_vals.mode()[0]
                        df.at[idx, col] = cat_maps[col][code]
                    else:
                        df.at[idx, col] = X_train[col].mode()[0]

                    n_imputed += 1

            print(f"  {name}: {n_imputed} values imputed via kNN")

        knn_impute(X_train, X_tr_num_scaled, "Train")
        knn_impute(X_val,  X_v_num_scaled,  "Val")
        knn_impute(X_test, X_te_num_scaled, "Test")

    else:
        print("  Skipped (missing required columns)")

    return X_train, X_val, X_test

In [18]:
X_train_cleann, X_val_cleann, X_test_cleann = impute_missing_values_hybrid(
    X_train, X_val, X_test
)
X_train_cleanc, X_val_cleanc, X_test_cleanc = impute_knn_categorical(
    X_train_cleann, X_val_cleann, X_test_cleann
)

HYBRID IMPUTATION PIPELINE

[5/6] NUMERICAL - IterativeImputer (MICE)
  Columns to impute: ['year', 'engineSize', 'mileage', 'mpg', 'tax', 'previousOwners']
  IterativeImputer applied successfully

[6/6] VALIDATION - checking logical limits
  Limits applied

IMPUTATION COMPLETED

Final missing values:
  Train: 4881
  Val:   2098
  Test:  3008

Columns with remaining NaNs in Train:
Brand           1071
model           1054
transmission    1604
fuelType        1152
dtype: int64

Columns with remaining NaNs in Val:
Brand           450
model           463
transmission    659
fuelType        526
dtype: int64

Columns with remaining NaNs in Test:
Brand           649
model           650
transmission    971
fuelType        738
dtype: int64

[3.25/6] kNN CATEGORICAL IMPUTATION (train-only)
  Train: 4881 values imputed via kNN
  Val: 2098 values imputed via kNN
  Test: 3008 values imputed via kNN


In [19]:
X_train=X_train_cleanc
X_val=X_val_cleanc
X_test=X_test_cleanc

In [20]:
X_train['car_age']=2025-X_train['year']
X_val['car_age']=2025-X_val['year']
X_test['car_age']=2025-X_test['year']
X_train['mileage_per_year'] = X_train['mileage'] / (X_train['car_age'] + 1)
X_val['mileage_per_year'] = X_val['mileage'] / (X_val['car_age'] + 1)
X_test['mileage_per_year'] = X_test['mileage'] / (X_test['car_age'] + 1)
eco_fuels = ['electric', 'hybrid']
X_train['is_eco'] = X_train['fuelType'].str.lower().isin(eco_fuels).astype(int)
X_val['is_eco'] = X_val['fuelType'].str.lower().isin(eco_fuels).astype(int)
X_test['is_eco'] = X_test['fuelType'].str.lower().isin(eco_fuels).astype(int)
luxury_brands = ["bmw", "audi", "mercedes"]
X_train["is_luxury"] = X_train["Brand"].str.lower().isin(luxury_brands).astype(int)
X_val["is_luxury"] = X_val["Brand"].str.lower().isin(luxury_brands).astype(int)
X_test["is_luxury"] = X_test["Brand"].str.lower().isin(luxury_brands).astype(int)
mileage_bins = [0, 10000, 50000, 100000, 150000, np.inf]
mileage_labels = ['0-10k', '10k-50k', '50k-100k', '100k-150k', '150k+']

X_train['mileage_bin'] = pd.cut(
    X_train['mileage'], bins=mileage_bins, labels=mileage_labels, include_lowest=True
)
X_val['mileage_bin'] = pd.cut(
    X_val['mileage'], bins=mileage_bins, labels=mileage_labels, include_lowest=True
)

X_test['mileage_bin'] = pd.cut(
    X_test['mileage'], bins=mileage_bins, labels=mileage_labels, include_lowest=True
)
X_train["tax_to_engine_ratio"] = X_train["tax"] / X_train["engineSize"].replace(0, np.nan)
X_val["tax_to_engine_ratio"] = X_val["tax"] / X_val["engineSize"].replace(0, np.nan)
X_test["tax_to_engine_ratio"] = X_test["tax"] / X_test["engineSize"].replace(0, np.nan)

In [21]:
X_train_num = X_train.select_dtypes(include=np.number).set_index(X_train.index).drop(['has_reportedDamage', 'is_eco', 'is_luxury'], axis=1)
X_train_cat = X_train.select_dtypes(exclude=np.number).set_index(X_train.index)


# Repeat for Validation
X_val_num = X_val.select_dtypes(include=np.number).set_index(X_val.index).drop([ 'has_reportedDamage', 'is_eco', 'is_luxury'], axis=1)
X_val_cat = X_val.select_dtypes(exclude=np.number).set_index(X_val.index)

# Repeat for Test
X_test_num = X_test.select_dtypes(include=np.number).set_index(X_test.index).drop(['has_reportedDamage', 'is_eco', 'is_luxury'], axis=1)
X_test_cat = X_test.select_dtypes(exclude=np.number).set_index(X_test.index)


X_train_binary=X_train[[ 'has_reportedDamage', 'is_eco', 'is_luxury']]
X_val_binary=X_val[['has_reportedDamage', 'is_eco', 'is_luxury']]
X_test_binary=X_test[['has_reportedDamage', 'is_eco', 'is_luxury']]

# Update metric features
cat_features=cat_features+['mileage_bin']
metric_features=X_train.columns.drop(cat_features).tolist()
print(metric_features)

['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'has_reportedDamage', 'car_age', 'mileage_per_year', 'is_eco', 'is_luxury', 'tax_to_engine_ratio']


In [22]:
#call function
scaler = RobustScaler()

#fit to training data
scaler.fit(X_train_num)

#transform the data
X_train_num_scaled = scaler.transform(X_train_num) # this will return an array

#show results
X_train_num_scaled

array([[ 0.66666667, -0.59681559,  0.        , ..., -0.66666667,
        -0.69853242,  0.39224138],
       [ 0.        ,  0.07491778, -6.25      , ...,  0.        ,
         0.06101524, -1.09913793],
       [ 0.66666667, -0.56657576,  0.        , ..., -0.66666667,
        -0.65255015, -0.04525862],
       ...,
       [-0.33333333,  1.7864763 , -1.        , ...,  0.33333333,
         1.79115739, -0.2262931 ],
       [-0.33333333,  0.59637443, -6.25      , ...,  0.33333333,
         0.52440078, -0.99568966],
       [ 0.66666667, -0.50870298,  0.        , ..., -0.66666667,
        -0.56454961,  1.26724138]])

In [23]:
# Convert the array to a pandas dataframe
X_train_num_scaled = pd.DataFrame(X_train_num_scaled, columns = X_train_num.columns).set_index(X_train.index)
X_train_num_scaled

,year,mileage,tax,mpg,engineSize,previousOwners,car_age,mileage_per_year,tax_to_engine_ratio
carID,,,,,,,,,
46299,0.666667,-0.596816,0.000000,-0.624204,-0.125,-1.0,-0.666667,-0.698532,0.392241
69620,0.000000,0.074918,-6.250000,0.382166,-0.250,-2.0,0.000000,0.061015,-1.099138
68925,0.666667,-0.566576,0.000000,-0.993631,0.500,-1.0,-0.666667,-0.652550,-0.045259
40663,-0.333333,1.677789,-7.111741,5.232227,0.500,0.0,0.333333,1.675470,-1.332729
25370,1.000000,-0.696078,0.341316,0.180332,-0.750,-1.0,-1.000000,-0.848453,1.390821
...,...,...,...,...,...,...,...,...,...
41593,0.666667,-0.552980,0.044594,-1.579701,1.750,1.0,-0.666667,-0.631876,-0.477377
19620,0.666667,-0.236504,0.000000,-1.127389,0.500,-1.0,-0.666667,-0.150648,-0.045259
66650,-0.333333,1.786476,-1.000000,0.222930,0.500,1.0,0.333333,1.791157,-0.226293


In [24]:
X_val_num_scaled = scaler.transform(X_val_num)
X_val_num_scaled = pd.DataFrame(X_val_num_scaled, columns = X_val_num.columns).set_index(X_val.index)
X_val_num_scaled.head()

X_test_num_scaled = scaler.transform(X_test_num)
X_test_num_scaled = pd.DataFrame(X_test_num_scaled, columns = X_test_num.columns).set_index(X_test.index)
X_test_num_scaled.head()

,year,mileage,tax,mpg,engineSize,previousOwners,car_age,mileage_per_year,tax_to_engine_ratio
carID,,,,,,,,,
89856,1.666667,0.531162,3.00,-0.885350,0.000,0.0,-1.666667,2.420822,0.961746
106581,0.000000,0.173394,0.25,-1.095541,0.500,-1.0,0.000000,0.177481,0.000000
80886,-0.333333,0.775487,-1.00,-0.254777,-0.125,-1.0,0.333333,0.715050,0.150862
100174,0.666667,-0.478182,0.00,-0.719745,-0.500,-2.0,-0.666667,-0.518140,0.829741
81376,0.666667,-0.336809,0.25,-0.254777,0.500,1.0,-0.666667,-0.303170,0.000000


In [25]:
low_card_features = ['transmission', 'fuelType', 'mileage_bin']
high_card_features = ['model','Brand']

# ONE-HOT ENCODING
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform
ohe_train = ohe.fit_transform(X_train_cat[low_card_features])
ohe_val = ohe.transform(X_val_cat[low_card_features])
ohe_test = ohe.transform(X_test_cat[low_card_features])

# DataFrames
ohe_columns = ohe.get_feature_names_out(low_card_features)
ohe_train_df = pd.DataFrame(ohe_train, columns=ohe_columns, index=X_train_cat.index)
ohe_val_df = pd.DataFrame(ohe_val, columns=ohe_columns, index=X_val_cat.index)
ohe_test_df = pd.DataFrame(ohe_test, columns=ohe_columns, index=X_test_cat.index)


# TARGET ENCODING
te = TargetEncoder(smooth=12, target_type="continuous")

# 'Brand'
te_train_brand = te.fit_transform(X_train_cat[['Brand']], y_train)
te_val_brand = te.transform(X_val_cat[['Brand']])
te_test_brand = te.transform(X_test_cat[['Brand']])

# Dataframes
te_train_df = pd.DataFrame(te_train_brand, columns=['Brand_Encoded'], index=X_train_cat.index)
te_val_df = pd.DataFrame(te_val_brand, columns=['Brand_Encoded'], index=X_val_cat.index)
te_test_df = pd.DataFrame(te_test_brand, columns=['Brand_Encoded'], index=X_test_cat.index)


# 'model'
te_model = TargetEncoder(smooth=50, target_type="continuous") #higher smoothing due to high cardinality
te_train_model = te_model.fit_transform(X_train_cat[['model']], y_train)
te_val_model = te_model.transform(X_val_cat[['model']])
te_test_model = te_model.transform(X_test_cat[['model']])

#Dataframes
te_train_model_df = pd.DataFrame(te_train_model, columns=['model_Encoded'], index=X_train_cat.index)
te_val_model_df = pd.DataFrame(te_val_model, columns=['model_Encoded'], index=X_val_cat.index)
te_test_model_df = pd.DataFrame(te_test_model, columns=['model_Encoded'], index=X_test_cat.index)

#joining all encoded categorical features
X_train_cat_encoded = pd.concat([ohe_train_df, te_train_df, te_train_model_df], axis=1)
X_val_cat_encoded = pd.concat([ohe_val_df, te_val_df, te_val_model_df], axis=1)
X_test_cat_encoded = pd.concat([ohe_test_df, te_test_df, te_test_model_df], axis=1)






In [26]:
X_train_final = pd.concat([X_train_num_scaled, X_train_cat_encoded, X_train_binary], axis=1)
X_val_final = pd.concat([X_val_num_scaled, X_val_cat_encoded,X_val_binary], axis=1)
X_test_final = pd.concat([X_test_num_scaled, X_test_cat_encoded,X_test_binary], axis=1)

In [27]:
X_train_final.columns

Index(['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners',
       'car_age', 'mileage_per_year', 'tax_to_engine_ratio',
       'transmission_Automatic', 'transmission_Manual',
       'transmission_Semi-Auto', 'fuelType_Diesel', 'fuelType_Electric',
       'fuelType_Hybrid', 'fuelType_Petrol', 'mileage_bin_0-10k',
       'mileage_bin_100k-150k', 'mileage_bin_10k-50k', 'mileage_bin_150k+',
       'mileage_bin_50k-100k', 'Brand_Encoded', 'model_Encoded',
       'has_reportedDamage', 'is_eco', 'is_luxury'],
      dtype='object')

In [28]:
X_train_selected = X_train_final.drop(['fuelType_Diesel', 'fuelType_Electric',
       'fuelType_Hybrid', 'fuelType_Petrol','mileage_per_year','has_reportedDamage','previousOwners','tax'], axis=1)
X_val_selected = X_val_final.drop(['fuelType_Diesel', 'fuelType_Electric',
       'fuelType_Hybrid', 'fuelType_Petrol','mileage_per_year','has_reportedDamage','previousOwners','tax'], axis=1)
X_test_selected = X_test_final.drop(['fuelType_Diesel', 'fuelType_Electric',
       'fuelType_Hybrid', 'fuelType_Petrol','mileage_per_year','has_reportedDamage','previousOwners','tax'], axis=1)


In [29]:
X_train_val=pd.concat([X_train_selected, X_val_selected], axis=0)
y_train_val=pd.concat([y_train, y_val], axis=0)

In [30]:
X_train_val

,year,mileage,mpg,engineSize,car_age,tax_to_engine_ratio,transmission_Automatic,transmission_Manual,transmission_Semi-Auto,mileage_bin_0-10k,mileage_bin_100k-150k,mileage_bin_10k-50k,mileage_bin_150k+,mileage_bin_50k-100k,Brand_Encoded,model_Encoded,is_eco,is_luxury
carID,,,,,,,,,,,,,,,,,,
46299,0.666667,-0.596816,-0.624204,-0.125000,-0.666667,0.392241,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,24423.828717,23552.735359,0,1
69620,0.000000,0.074918,0.382166,-0.250000,0.000000,-1.099138,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,16856.870038,11567.281213,0,0
68925,0.666667,-0.566576,-0.993631,0.500000,-0.666667,-0.045259,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,16883.531098,11495.865125,0,0
40663,-0.333333,1.677789,5.232227,0.500000,0.333333,-1.332729,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,24592.639649,23602.295737,1,1
25370,1.000000,-0.696078,0.180332,-0.750000,-1.000000,1.390821,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,12549.566846,13413.766088,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10446,-3.333333,5.997594,0.140127,0.500000,3.333333,-0.226293,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,21783.973107,16275.530643,0,1
677,-0.666667,1.584343,-0.885350,1.750000,0.666667,0.211207,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,23033.559080,29066.910922,0,1
13167,0.000000,0.979947,0.656051,0.500000,0.000000,-0.045259,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,21783.973107,19360.321327,0,1


In [31]:
best_model_rf = RandomForestRegressor(
                    n_estimators=1200,
                    max_depth=30,
                    min_samples_split=5,
                    min_samples_leaf=2,
                    max_features=0.5,
                    random_state=42,
                    n_jobs=-1
                )

In [32]:
best_model_rf.fit(X_train_selected,y_train)

,n_estimators,1200
,criterion,'squared_error'
,max_depth,30
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [33]:
def compute_metrics(model, X, y, split):
    """
    Compute evaluation metrics for a regression model.
    Args:
        model: Fitted regression model.
        X: Features for prediction.
        y: True target values.
        split: Data split identifier (e.g., 'train', 'val', 'test').
    Returns:
        Dictionary of evaluation metrics.
    """
    y_pred = model.predict(X)
    return {
        "split": split,
        "MAE": mean_absolute_error(y, y_pred),
        "MedAE": median_absolute_error(y, y_pred),
        "RMSE": root_mean_squared_error(y, y_pred),
        "MAPE": mean_absolute_percentage_error(y, y_pred),
        "R2": r2_score(y, y_pred),
    }

In [34]:
metrics_df = pd.DataFrame([
    compute_metrics(best_model_rf, X_train_selected, y_train, "train"),
    compute_metrics(best_model_rf, X_val_selected,   y_val,   "val"),
]).set_index("split")

display(metrics_df)

,MAE,MedAE,RMSE,MAPE,R2
split,,,,,
train,777.779934,480.275006,1433.621743,0.048007,0.978512
val,1360.984797,861.339721,2248.643983,0.084440,0.945536


In [35]:
best_model_rf.fit(X_train_val,y_train_val)

,n_estimators,1200
,criterion,'squared_error'
,max_depth,30
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [36]:
y_test = best_model_rf.predict(X_test_selected)
y_test= pd.Series(y_test, name="price", index=X_test.index)
y_test.to_csv("y_test.csv",index=True)